# Dataset Exploration

In [ ]:
# select dataset
SUBJECT = "ants"
VERSION = "v2"

### Import Libraries

In [ ]:
import sys
sys.path.insert(0, '..')
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from PIL import Image
from IPython.display import display, HTML

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("darkgrid")
sns.set_palette("husl")
%matplotlib inline

## 1. Import Dataset

#### 1.1 Load annotations

In [ ]:
# Load annotations
annotations_path = Path(f"../../dataset/{SUBJECT}/{VERSION}/annotations.csv")

if not annotations_path.exists():
    print(f"❌ Annotations not found: {annotations_path}")
    print(f"Run: python -m src.dataset.get_annotations --subject {SUBJECT} --version {VERSION}")
else:
    df_frames = pd.read_csv(annotations_path)
    print(f"✓ Loaded {len(df_frames):,} frames from {len(df_frames['observation_id'].unique())} observations")
    print(f"\nColumns: {list(df_frames.columns)}")
    print(f"\nDataset shape: {df_frames.shape}")
    display(df_frames.head())

In [ ]:
# basic statistics per frame
print("=" * 60)
print("DATASET STATISTICS")
print("=" * 60)

# general
print(f"\nTotal observations: {df_frames['observation_id'].nunique()}")
print(f"Total frames: {len(df_frames):,}")
print(f"Frames per observation: {len(df_frames) / df_frames['observation_id'].nunique():.0f} ± {df_frames.groupby('observation_id').size().std():.1f}")
m = df_frames.groupby("observation_id").apply(lambda g: len(g) / g["fps"].iloc[0] / 60)
total_minutes = (df_frames.groupby("observation_id").apply(lambda g: len(g) / g["fps"].iloc[0]).sum()) / 60
d, h, mi = divmod(total_minutes, 24*60)[0], *divmod(divmod(total_minutes, 24*60)[1], 60)
print(f"Minutes per observation: {m.mean():.1f} ± {m.std():.1f}")
print(f"Total duration: {total_minutes:.1f} minutes ({int(d)}d {int(h)}h {int(mi)}m)")



# treatment
print(f"\nTreatment:")
treatment_pct = (df_frames['T'].value_counts().sort_index() / len(df_frames) * 100).round(2)
for treatment, pct in treatment_pct.items():
    print(f"  {treatment}: {pct}%")

# covariate 
w_cols = [c for c in df_frames.columns if c.startswith('W_')]
if w_cols:
    print(f"\nCovariates ({len(w_cols)}): {', '.join([c.replace('W_', '') for c in w_cols])}")

# outcome 
y_cols = [c for c in df_frames.columns if c.startswith('Y_')]
if y_cols:
    print(f"\nOutcomes ({len(y_cols)}): {', '.join([c.replace('Y_', '') for c in y_cols])}")
    for col in y_cols:
        print(f"\n{col}:")
        outcome_pct = (df_frames[col].value_counts().sort_index() / len(df_frames) * 100).round(2)
        for value, pct in outcome_pct.items():
            print(f"  {value}: {pct}%")

#### 1.2 Aggregate Observations

In [ ]:
# basic statistics per observation
y_cols = [c for c in df_frames.columns if c.startswith('Y_')]
w_cols = [c for c in df_frames.columns if c.startswith('W_')]

agg_dict = {
    'n_frames': ('frame_idx', 'count'),
    'T': ('T', 'first'),
}

for col in w_cols:
    # assuming covariates do not change over observation
    agg_dict[col] = (col, 'first') 
for col in y_cols:
    agg_dict[col] = (col, 'mean')

df_obs = df_frames.groupby('observation_id').agg(**agg_dict)

display(df_obs)
# display(df_obs.describe())

In [ ]:
df_obs_T = df_obs.groupby('T')[y_cols].mean()

n_T   = df_obs.groupby('T')[y_cols].count()
std_T = df_obs.groupby('T')[y_cols].std(ddof=1)
se_T  = std_T / np.sqrt(n_T)
ci95_T = 1.96 * se_T

means = df_obs_T.T                      # shape: (len(y_cols), n_treatments)
errs  = ci95_T.T.reindex_like(means)    # ensure exact same shape + labels

fig, ax = plt.subplots(figsize=(10, 6))
means.plot(
    kind='bar',
    ax=ax,
    width=0.8,
    color=["#47E3FF", "#77DD77", "#F49AC2"],
    yerr=errs,          
    capsize=4,
)

ax.set_xlabel('Grooming')
ax.set_ylabel('Time (%)')
ax.set_xticklabels(['Blue to Focal', 'Yellow to Focal'], rotation=0, ha='right')
ax.legend(title='Treatment', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(axis='y', alpha=0.3)

#display(df_obs_T)

#### 1.3 Time decay

In [ ]:
# minute column
df_frames['minute'] = (df_frames['frame_idx'] // (60 * df_frames['fps'])).astype(int)

# (1) aggregate within each observation: mean over frames in the same minute
df_rep_min = (
    df_frames
    .groupby(['observation_id', 'minute', 'T'], as_index=False)[y_cols]
    .mean()
)

# x-limits based on what we actually plot
xmin = int(df_rep_min['minute'].min())
xmax = int(df_rep_min['minute'].max())

fig, axes = plt.subplots(1, len(y_cols), figsize=(15, 5), sharey=True)

for i, col in enumerate(y_cols):
    ax = axes[i]
    sns.lineplot(
        data=df_rep_min,
        x='minute',
        y=col,
        hue='T',
        ax=ax,
        marker='o',
        estimator='mean',      # mean across observations
        errorbar=('ci', 95),   # 95% CI across observations
        # n_boot=2000,         # uncomment for more stable bootstrap CI
    )
    ax.set_title("Blue to Focal" if col == 'Y_B2F' else "Yellow to Focal")
    ax.set_xlabel('Minute')
    ax.set_ylabel('Time (%)' if i == 0 else '')
    ax.legend(title='Treatment')
    ax.set_xlim(xmin, xmax)

plt.tight_layout()

## 2. Visualize

#### 2.1 Plot frames with annotations

In [ ]:
# Select random samples (stratified by treatment)
n_samples_per_treatment = 4
samples = df_frames.groupby('T').apply(lambda x: x.sample(n_samples_per_treatment, random_state=42)).reset_index(drop=True)

# Create figure
n_samples = len(samples)
n_cols = n_samples_per_treatment
n_rows = (n_samples + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5 * n_rows))
axes = axes.flatten() if n_samples > 1 else [axes]

for idx, (_, row) in enumerate(samples.iterrows()):
    if idx >= len(axes):
        break
    
    # Load image (frame_path is relative from workspace root, adjust for notebook location)
    img_path = Path('..') / 'dataset' / row['frame_path']
    if img_path.exists():
        img = Image.open(img_path)
        axes[idx].imshow(img)
        
        # Create title with annotations
        title = f"Obs: {row['observation_id']}, Frame: {row['frame_idx']}, T={row['T']}"
        if y_cols:
            outcomes = ", ".join([f"{col.replace('Y_', '')}={row[col]}" for col in y_cols])
            title += f"\n{outcomes}"
        
        axes[idx].set_title(title, fontsize=10)
    else:
        axes[idx].text(0.5, 0.5, 'Image not found', ha='center', va='center')
        axes[idx].set_title(f"Missing: {img_path}", fontsize=9, color='red')
    
    axes[idx].axis('off')

# Hide unused subplots
for idx in range(n_samples, len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

#### 2.1 Plot clip with annotations

In [ ]:
import cv2
import subprocess
from IPython.display import Video

def create_clip(observation_id, start_frame=0, n_frames=30, fps=10, annotate=True, output_path=None):
    """
    Create a video clip from frames with optional real-time annotations.
    
    Args:
        observation_id: Observation to extract from
        start_frame: Starting frame index (0-indexed)
        n_frames: Number of frames to include
        fps: Frames per second for output video
        annotate: Whether to add annotations to frames (default: True)
        output_path: Where to save (default: temp file)
    
    Returns:
        Path to created video file
    """
    # Get frames for this observation
    obs_df_frames = df_frames[df_frames['observation_id'] == observation_id].sort_values('frame_idx')
    obs_df_frames = obs_df_frames.iloc[start_frame:start_frame + n_frames]
    
    if len(obs_df_frames) == 0:
        print(f"No frames found for observation {observation_id}")
        return None
    
    print(f"Creating clip from {len(obs_df_frames)} frames at {fps} fps...")
    
    # Load frames
    frames = []
    frame_size = None
    
    for _, row in obs_df_frames.iterrows():
        img_path = Path('..') / 'dataset' / row['frame_path']
        if img_path.exists():
            # Load with opencv (BGR format)
            img = cv2.imread(str(img_path))
            if img is None:
                print(f"Warning: Could not load {img_path}")
                continue
            
            # Set frame size from first frame
            if frame_size is None:
                frame_size = (img.shape[1], img.shape[0])  # (width, height)
            
            # Resize if needed
            if (img.shape[1], img.shape[0]) != frame_size:
                img = cv2.resize(img, frame_size)
            
            # Add annotations if requested
            if annotate:
                annotated_img = img.copy()
                
                # Set up text properties
                font = cv2.FONT_HERSHEY_SIMPLEX
                font_scale = 0.5
                thickness = 1
                line_height = 20
                
                # Get treatment info
                treatment = row['T']
                
                # Create annotation text
                annotations = [
                    f"Obs: {observation_id} | T: {treatment}",
                    f"Frame: {row['frame_idx']}",
                ]
                
                # Add outcome annotations
                for col in y_cols:
                    outcome_name = col.replace('Y_', '')
                    outcome_value = row[col]
                    annotations.append(f"{outcome_name}: {outcome_value}")
                
                # Draw semi-transparent background for text
                overlay = annotated_img.copy()
                bg_height = len(annotations) * line_height + 10
                cv2.rectangle(overlay, (5, 5), (frame_size[0] - 5, bg_height), (0, 0, 0), -1)
                cv2.addWeighted(overlay, 0.6, annotated_img, 0.4, 0, annotated_img)
                
                # Draw text
                y_offset = 20
                for text in annotations:
                    cv2.putText(annotated_img, text, (10, y_offset), 
                               font, font_scale, (255, 255, 255), thickness, cv2.LINE_AA)
                    y_offset += line_height
                
                frames.append(annotated_img)
            else:
                frames.append(img)
        else:
            print(f"Warning: Frame not found: {img_path}")
    
    if len(frames) == 0:
        print("No valid frames found")
        return None
    
    # Save video
    if output_path is None:
        annot_suffix = "_annotated" if annotate else ""
        output_path = f"../outputs/{SUBJECT}/{VERSION}/clips/{observation_id}_{start_frame}_{n_frames}{annot_suffix}.mp4"
    
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    
    temp_path = str(output_path).replace('.mp4', '_temp.avi')
    
    try:
        print(f"Saving {len(frames)} frames to temporary file...")
        # Use MJPEG codec which is more universally available
        fourcc = cv2.VideoWriter_fourcc(*'MJPG')
        out = cv2.VideoWriter(temp_path, fourcc, fps, frame_size)
        
        for frame in frames:
            out.write(frame)
        out.release()
        
        # Try converting to MP4 using available encoders
        print(f"Converting to MP4...")
        
        # Try different encoder options
        encoders = [
            ['ffmpeg', '-y', '-i', temp_path, '-vcodec', 'libopenh264', '-pix_fmt', 'yuv420p', str(output_path)],
            ['ffmpeg', '-y', '-i', temp_path, '-vcodec', 'mpeg4', '-pix_fmt', 'yuv420p', str(output_path)],
        ]
        
        success = False
        for encoder_cmd in encoders:
            result = subprocess.run(encoder_cmd, capture_output=True, text=True)
            if result.returncode == 0:
                Path(temp_path).unlink()
                print(f"✓ Saved clip to {output_path}")
                success = True
                break
        
        if not success:
            # Fall back to keeping the AVI
            import shutil
            avi_path = str(output_path).replace('.mp4', '.avi')
            shutil.move(temp_path, avi_path)
            print(f"✓ Saved clip to {avi_path} (MJPEG/AVI format)")
            return avi_path
        
    except Exception as e:
        print(f"Error: {e}")
        if Path(temp_path).exists():
            Path(temp_path).unlink()
        return None
    
    return output_path

# Create a sample clip with annotations
clip_path = create_clip(observation_id = "a2", 
                        start_frame = 100, 
                        n_frames = 400, 
                        fps = fps,
                        annotate = True)  

if clip_path:
    print(f"\n▶ Video ready for playback:")
    display(Video(str(clip_path), embed=True, width=600))

## 3. Hugging Face dataset